# Gradient Flow Analysis

Analyzes how gradients propagate through the transformer, detecting vanishing/exploding gradients and attributing gradient magnitude to specific components.

**References:**
- Pascanu et al. (2013) "On the Difficulty of Training Recurrent Neural Networks"
- Zhang et al. (2019) "Improving Deep Transformer with Depth-Scaled Initialization"

## Why This Matters

Gradient flow analysis traces how learning signals propagate backward through the network. Gradient norms, saturation, and LayerNorm effects reveal which components receive strong training signals and which are in gradient dead zones — connecting architecture to trainability.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.gradient_flow import (
    gradient_norm_by_layer,
    component_gradient_attribution,
    gradient_saturation_analysis,
    layernorm_gradient_effect,
    per_head_gradient_sensitivity,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Gradient norms by layer
gnorms = gradient_norm_by_layer(model, tokens)
print(f"Vanishing: {gnorms['vanishing']}, ratio: {gnorms['gradient_ratio']:.4f}")
for i, norm in enumerate(gnorms['layer_grad_norms']):
    label = 'embed' if i == 0 else f'layer {i-1}'
    print(f"  {label}: grad_norm={norm:.6f}")

In [ ]:
# 2. Component gradient attribution
cga = component_gradient_attribution(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: attn={cga['attn_grad_norms'][l]:.4f}, "
          f"mlp={cga['mlp_grad_norms'][l]:.4f}, "
          f"attn_frac={cga['attn_fraction'][l]:.3f}, "
          f"dominant={cga['dominant_component_per_layer'][l]}")

In [ ]:
# 3. Gradient saturation
sat = gradient_saturation_analysis(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: saturation={sat['layer_saturation'][l]:.4f}, "
          f"mean|pre|={sat['pre_activation_means'][l]:.4f}, "
          f"frac_sat={sat['fraction_saturated'][l]:.4f}")

In [ ]:
# 4. LayerNorm gradient effect
ln_eff = layernorm_gradient_effect(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: in_norm={ln_eff['input_norms'][l]:.4f}, "
          f"out_norm={ln_eff['output_norms'][l]:.4f}, "
          f"compression={ln_eff['compression_ratio'][l]:.4f}")

In [ ]:
# 5. Per-head gradient sensitivity
for l in range(cfg.n_layers):
    sens = per_head_gradient_sensitivity(model, tokens, layer=l)
    vals = [f'{s:.4f}' for s in sens['head_sensitivities']]
    print(f"Layer {l}: {vals}, most={sens['most_sensitive_head']}, ratio={sens['sensitivity_ratio']:.2f}")